# Veritas-Clin — End-to-End Run (Colab T4)

Evidence-grounded clinical extraction: zero-shot LLM extraction → NLI verification →
calibrated abstention (threshold learned on dev, reported on test).

Before running: upload/clone the `veritas-clin` repo into Colab's working directory
(Runtime type: T4 GPU).

In [ ]:
!pip install -q accelerate fastapi pydantic
!pip install -q -e .

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime"
print(torch.cuda.get_device_name(0))

## 1. Corpus (synthetic, ground-truth annotated, planted hallucination traps)

In [ ]:
from veritas_clin.data_gen import ensure_corpus

corpus = ensure_corpus()
print(corpus[0]["text"])
print("ground truth:")
for g in corpus[0]["gt"]:
    print(" -", g["type"], "->", g["value"])

## 2. Models: zero-shot extractor + NLI verifier

In [ ]:
from veritas_clin.extract import HuggingFaceExtractor
from veritas_clin.verify import Verifier

extractor = HuggingFaceExtractor()  # Qwen2.5-3B-Instruct, fp16
verifier = Verifier()               # DeBERTa-v3 MNLI checkpoint

## 3. Full evaluation: dev-split threshold, test-split metrics

In [ ]:
from veritas_clin.eval import run_eval

metrics = run_eval(extractor_kind="hf", use_nli=True, n_notes=40, seed=7)
import json
print(json.dumps(metrics, indent=2))

In [ ]:
from IPython.display import Image, display

display(Image("artifacts/plots/calibration.png"))
display(Image("artifacts/plots/risk_coverage.png"))

## 4. Demo: decisions on a single note

In [ ]:
from veritas_clin.pipeline import assign_decisions, extract_and_verify

policy = json.load(open("artifacts/policy.json"))
record = corpus[0]
claims = extract_and_verify(extractor, verifier, record["text"])
assign_decisions(claims, policy["threshold"], policy["low_frac"])

print("threshold tau =", policy["threshold"], "\n")
for c in claims:
    print(f"{c.decision.upper():17} {c.type.value:10} {c.value!r:42} "
          f"p_entail={c.p_entail:.3f}  score={c.score:.3f}")